## 手写GRU
#### 更新门
$ z_t=\sigma(W_xz \cdot x_t+W_hz \cdot h_{t-1}+b_z) $
#### 重置门
$ r_t=\sigma(W_xr \cdot x_t+W_hr \cdot h_{t-1}+b_r) $
#### 候选隐状态
$ h_t=tanh(W_xh \cdot x_t+W_hh \cdot (r_t \cdot h_{t-1})+b_h) $
#### 输出门
$ h_t=z_t \cdot h_t+(1-z_t) \cdot h_{t-1} $

In [ ]:
from torch import nn
import torch


class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.W_xz = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hz = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.W_xr = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hr = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.b_z = nn.Parameter(torch.zeros(hidden_size))
        self.b_r = nn.Parameter(torch.zeros(hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h_prev):
        # 更新门
        z_t = torch.sigmoid(torch.mm(x, self.W_xz) + torch.mm(h_prev, self.W_hz) + self.b_z)
        # 重置门
        r_t = torch.sigmoid(torch.mm(x, self.W_xr) + torch.mm(h_prev, self.W_hr) + self.b_r)
        # 候选隐状态
        h_candidate = torch.tanh(torch.mm(x, self.W_xh) + torch.mm(r_t * h_prev, self.W_hh) + self.b_h)
        # 最终隐状态
        h_t = z_t * h_candidate + (1 - z_t) * h_prev
        return h_t


class GRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)

    def forward(self, x, hidden=None):
        seq_len, batch_size, input_size = x.shape
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size)
        outputs = []
        for t in range(seq_len):
            hidden = self.cell(x[t], hidden)
            outputs.append(hidden)
        return outputs, hidden





In [ ]:
input_size = 32
hidden_size = 5
batch_size = 1
seq_len = 2


model = GRU(input_size, hidden_size)
x = torch.randn(seq_len, batch_size, input_size)
outputs, hidden = model(x)

print(outputs)
print(hidden)


## GRU文本生成

In [7]:
text = """
臣密言：臣以险衅，夙遭闵凶。生孩六月，慈父见背；行年四岁，舅夺母志。
祖母刘愍臣孤弱，躬亲抚养。
臣少多疾病，九岁不行，零丁孤苦，至于成立。
既无伯叔，终鲜兄弟，门衰祚薄，晚有儿息。
外无期功强近之亲，内无应门五尺之僮，茕茕孑立，形影相吊。
而刘夙婴疾病，常在床蓐，臣侍汤药，未曾废离。
"""

words = set(text)
vocab_size = len(words)
word_to_index = {word: i for i, word in enumerate(words)}
index_to_word = {i: word for i, word in enumerate(words)}

print(word_to_index)

{'弱': 0, '孑': 1, '蓐': 2, '影': 3, '近': 4, '伯': 5, '养': 6, '六': 7, '汤': 8, '苦': 9, '薄': 10, '慈': 11, '\n': 12, '兄': 13, '零': 14, '背': 15, '父': 16, '。': 17, '门': 18, '无': 19, '应': 20, '母': 21, '相': 22, '吊': 23, '尺': 24, '岁': 25, '抚': 26, '至': 27, '疾': 28, '强': 29, '言': 30, '：': 31, '在': 32, '孤': 33, '孩': 34, '凶': 35, '生': 36, '四': 37, '愍': 38, '年': 39, '曾': 40, '祚': 41, '侍': 42, '遭': 43, '弟': 44, '九': 45, '祖': 46, '不': 47, '婴': 48, '常': 49, '以': 50, '叔': 51, '密': 52, '见': 53, '儿': 54, '未': 55, '鲜': 56, '床': 57, '离': 58, '丁': 59, '于': 60, '臣': 61, '既': 62, '期': 63, '立': 64, '成': 65, '闵': 66, '药': 67, '刘': 68, '；': 69, '僮': 70, '终': 71, '志': 72, '五': 73, '行': 74, '少': 75, '月': 76, '病': 77, '晚': 78, '内': 79, '之': 80, '衰': 81, '躬': 82, '茕': 83, '息': 84, '而': 85, '，': 86, '外': 87, '舅': 88, '亲': 89, '险': 90, '多': 91, '有': 92, '夺': 93, '衅': 94, '功': 95, '夙': 96, '形': 97, '废': 98}


In [9]:
from torch.utils.data import Dataset
import torch

SEQ_LEN = 5
BATCH_SIZE = 1
HIDDEN_SIZE = 128
EMBEDDING_SIZE = 128


class TextDataset(Dataset):
    def __init__(self, text, seq_len):
        self.text = text
        self.seq_len = seq_len
        self.data = [word_to_index[ch] for ch in text]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        input_seq = self.data[index:index + self.seq_len]
        target_seq = self.data[index + 1:index + self.seq_len + 1]
        return torch.tensor(input_seq), torch.tensor(target_seq)


dataset = TextDataset(text, SEQ_LEN)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(dataset.data)

[12, 61, 52, 30, 31, 61, 50, 90, 94, 86, 96, 43, 66, 35, 17, 36, 34, 7, 76, 86, 11, 16, 53, 15, 69, 74, 39, 37, 25, 86, 88, 93, 21, 72, 17, 12, 46, 21, 68, 38, 61, 33, 0, 86, 82, 89, 26, 6, 17, 12, 61, 75, 91, 28, 77, 86, 45, 25, 47, 74, 86, 14, 59, 33, 9, 86, 27, 60, 65, 64, 17, 12, 62, 19, 5, 51, 86, 71, 56, 13, 44, 86, 18, 81, 41, 10, 86, 78, 92, 54, 84, 17, 12, 87, 19, 63, 95, 29, 4, 80, 89, 86, 79, 19, 20, 18, 73, 24, 80, 70, 86, 83, 83, 1, 64, 86, 97, 3, 22, 23, 17, 12, 85, 68, 96, 48, 28, 77, 86, 49, 32, 57, 2, 86, 61, 42, 8, 67, 86, 55, 40, 98, 58, 17, 12]


In [12]:
import torch
import torch.nn as nn


class GRU(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True,num_layers=2)
        self.out_linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedding = self.embedding(x)
        outputs, hidden = self.gru(embedding, hidden)
        outputs = self.out_linear(outputs)
        return outputs, hidden


In [13]:
model = GRU(vocab_size, EMBEDDING_SIZE, HIDDEN_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    for i, (input_seq, target_seq) in enumerate(train_loader):
        output, _ = model(input_seq)
        loss = criterion(
            output.view(-1, vocab_size),
            target_seq.view(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if i % 10 == 0:
            print(f'Epoch [{epoch + 1}/100], Step [{i + 10}/{len(train_loader)}], Loss: {loss.item():.8f}')

Epoch [1/100], Step [10/140], Loss: 4.6179
Epoch [1/100], Step [20/140], Loss: 4.6110
Epoch [1/100], Step [30/140], Loss: 4.4070
Epoch [1/100], Step [40/140], Loss: 4.3916
Epoch [1/100], Step [50/140], Loss: 4.6752
Epoch [1/100], Step [60/140], Loss: 4.0334
Epoch [1/100], Step [70/140], Loss: 3.9876
Epoch [1/100], Step [80/140], Loss: 4.2701
Epoch [1/100], Step [90/140], Loss: 3.3564
Epoch [1/100], Step [100/140], Loss: 4.0599
Epoch [1/100], Step [110/140], Loss: 3.8108
Epoch [1/100], Step [120/140], Loss: 3.0500
Epoch [1/100], Step [130/140], Loss: 3.7223
Epoch [1/100], Step [140/140], Loss: 3.7492
Epoch [2/100], Step [10/140], Loss: 3.6246
Epoch [2/100], Step [20/140], Loss: 2.9368
Epoch [2/100], Step [30/140], Loss: 2.6878
Epoch [2/100], Step [40/140], Loss: 2.4082
Epoch [2/100], Step [50/140], Loss: 2.3009
Epoch [2/100], Step [60/140], Loss: 2.2979
Epoch [2/100], Step [70/140], Loss: 2.5104
Epoch [2/100], Step [80/140], Loss: 1.7153
Epoch [2/100], Step [90/140], Loss: 1.6585
Epoch 

In [14]:

model.eval()


def generate_text(context, step, temperature=0.8):
    words = [word for word in context]
    hidden = None
    for _ in range(step):
        input_seq = torch.tensor([word_to_index[word] for word in words[-1:]])
        input_seq = torch.LongTensor(input_seq)
        input_seq = input_seq.view(1, -1)

        with torch.no_grad():
            output, hidden = model(input_seq, hidden)
            last_output = output[0, -1, :]
            probs = torch.softmax(last_output / temperature, dim=-1)
            result_index = torch.multinomial(probs, 1).item()
            result = index_to_word[result_index]
            words.append(result)
    return ''.join(words)


print(generate_text('臣密言：', 5, 0.1))

臣密言：臣以险衅，
